In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv() 

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)

from __future__ import annotations

import argparse
import math
import os
import random
import time
from typing import Dict, List, Optional, Tuple

import matplotlib
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from rf_common import (Config, PdwSource, RunningStats, StateBuilder, add_data_args, build_action_table,
                       build_network, freeze_network, make_env, probe_source, source_from_args)

VARIANTS: Dict[str, Dict[str, bool]] = {
    "dueling":        {"recurrent": False, "double": False},   # non-recurrent control
    "recurrent":      {"recurrent": True, "double": False},    # GRU+attention, Dueling DQN
    "recurrent_ddqn": {"recurrent": True, "double": True},     # GRU+attention, Dueling Double DQN
}


class SequenceReplayBuffer:
    def __init__(self, capacity, seq_len, frame_dim, ekf_dim, rng):
        self.cap, self.rng, self.n, self.i = capacity, rng, 0, 0
        self.s = np.zeros((capacity, seq_len, frame_dim), np.float32)
        self.s2 = np.zeros_like(self.s)
        self.e = np.zeros((capacity, ekf_dim), np.float32)
        self.e2 = np.zeros_like(self.e)
        self.a = np.zeros(capacity, np.int64)
        self.r = np.zeros(capacity, np.float32)
        self.d = np.zeros(capacity, np.float32)

    def push(self, s, e, a, r, s2, e2, d):
        i = self.i
        self.s[i], self.e[i], self.a[i], self.r[i], self.s2[i], self.e2[i], self.d[i] = s, e, a, r, s2, e2, d
        self.i = (i + 1) % self.cap
        self.n = min(self.n + 1, self.cap)

    def sample(self, B):
        idx = self.rng.integers(0, self.n, B)
        return self.s[idx], self.e[idx], self.a[idx], self.r[idx], self.s2[idx], self.e2[idx], self.d[idx]

    def __len__(self):
        return self.n


class SchedulerAgent:
    def __init__(self, cfg: Config, num_actions: int, recurrent: bool, double: bool,
                 device: torch.device, rng: np.random.Generator):
        self.cfg, self.device, self.double, self.recurrent = cfg, device, double, recurrent
        self.num_actions = num_actions
        ekf_dim = 4 * cfg.num_bins if cfg.use_ekf else 0
        self.q_network = build_network(cfg, num_actions, recurrent).to(device)
        self.target_network = build_network(cfg, num_actions, recurrent).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=cfg.lr)
        self.buffer = SequenceReplayBuffer(cfg.buffer_capacity, cfg.seq_len, 2 * cfg.num_bins, ekf_dim, rng)
        self.learn_steps = 0

    def _t(self, x, dtype=torch.float32):
        return torch.as_tensor(x, dtype=dtype, device=self.device)

    def greedy_action(self, seq: np.ndarray, ekf: np.ndarray) -> int:
        self.q_network.eval()
        with torch.no_grad():
            q = self.q_network(self._t(seq).unsqueeze(0), self._t(ekf).unsqueeze(0) if ekf.size else None)
        self.q_network.train()
        return int(q.argmax(dim=1).item())

    def learn(self) -> Optional[float]:
        if len(self.buffer) < self.cfg.batch_size:
            return None
        s, e, a, r, s2, e2, d = self.buffer.sample(self.cfg.batch_size)
        s, s2 = self._t(s), self._t(s2)
        e_t, e2_t = (self._t(e), self._t(e2)) if e.shape[1] else (None, None)
        a = self._t(a, torch.int64).unsqueeze(1)
        r, d = self._t(r).unsqueeze(1), self._t(d).unsqueeze(1)

        q = self.q_network(s, e_t).gather(1, a)
        with torch.no_grad():
            if self.double:
                a2 = self.q_network(s2, e2_t).argmax(dim=1, keepdim=True)
                q2 = self.target_network(s2, e2_t).gather(1, a2)
            else:
                q2 = self.target_network(s2, e2_t).max(dim=1, keepdim=True)[0]
            y = r + self.cfg.gamma * q2 * (1.0 - d)
        loss = nn.functional.smooth_l1_loss(q, y)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), 10.0)
        self.optimizer.step()
        self.learn_steps += 1
        if self.learn_steps % self.cfg.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        return float(loss.item())

    def save_checkpoint(self, path: str, total_steps: int, ep: int, hist: Dict) -> None:
        """Saves full training state to resume later."""
        checkpoint = {
            "q_network": self.q_network.state_dict(),
            "target_network": self.target_network.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "learn_steps": self.learn_steps,
            "total_steps": total_steps,
            "completed_episodes": ep,
            "hist": hist,
        }
        torch.save(checkpoint, path)

    def load_checkpoint(self, path: str) -> Tuple[int, int, Dict]:
        """Loads training state and returns (total_steps, completed_episodes, hist)."""
        checkpoint = torch.load(path, map_location=self.device)
        self.q_network.load_state_dict(checkpoint["q_network"])
        self.target_network.load_state_dict(checkpoint["target_network"])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        self.learn_steps = checkpoint.get("learn_steps", 0)
        print(f"--> Successfully loaded checkpoint from '{path}'")
        return (
            checkpoint.get("total_steps", 0),
            checkpoint.get("completed_episodes", 0),
            checkpoint.get("hist", {"base_reward": [], "shaped_reward": [], "pd": [], "pfa": [], "scan_eff": [], "epsilon": [], "loss": []})
        )


class SurpriseTracker:
    def __init__(self, num_bins, decay=0.999, alpha=1.0):
        self.N, self.decay, self.alpha = num_bins, decay, alpha
        self.counts = np.zeros(num_bins)

    def surprisal(self, b: int) -> float:
        p = (self.counts[b] + self.alpha) / (self.counts.sum() + self.alpha * self.N)
        return float(np.clip(-math.log(p) / math.log(self.N), 0.0, 1.0))

    def update(self, b: int) -> None:
        self.counts *= self.decay
        self.counts[b] += 1.0


class RewardShaper:
    """final = env reward + anomaly_weight * surprisal(intercepted) - redundancy penalty."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.surprise = SurpriseTracker(cfg.num_bins, cfg.surprise_decay)
        self.reset_episode()

    def reset_episode(self):
        self.last_action, self.streak = None, 0

    def shape(self, base_reward: float, action_idx: int, intercepted_pdws: list):
        c = self.cfg
        anomaly = sum(c.anomaly_weight * self.surprise.surprisal(p["bin_idx"]) for p in intercepted_pdws)
        for p in intercepted_pdws:
            self.surprise.update(p["bin_idx"])
        if action_idx == self.last_action and not intercepted_pdws:
            self.streak += 1
        else:
            self.streak = 0
        self.last_action = action_idx
        redundancy = c.redundancy_weight * min(max(self.streak - c.redundancy_grace, 0), c.redundancy_cap)
        return base_reward + anomaly - redundancy, {"anomaly": anomaly, "redundancy": redundancy}


class EpsilonGuidedExploration:
    """Epsilon-greedy, linear decay; part of the exploratory picks target the stalest bins."""

    def __init__(self, cfg: Config, action_index: Dict[Tuple[int, ...], int], rng: np.random.Generator):
        # FIXED: Added 'rng' to the right side of the assignment
        self.cfg, self.idx, self.rng = cfg, action_index, rng
        self.num_actions = len(action_index)

    def epsilon(self, step: int) -> float:
        c = self.cfg
        return c.eps_start + min(1.0, step / c.eps_decay_steps) * (c.eps_end - c.eps_start)

    def explore(self, builder: StateBuilder) -> int:
        if self.rng.random() < self.cfg.guided_fraction:
            return self.idx[tuple(builder.stalest_bins(self.cfg.k_observed, self.rng))]
        return int(self.rng.integers(self.num_actions))


#Training loop

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train(cfg: Config, source: PdwSource, variant: str, seed: int, num_episodes: int,
          out_dir: str = "train_out", resume: bool = True, print_every: int = 10, verbose: bool = True) -> Dict:
    spec = VARIANTS[variant]
    set_seed(seed)
    rng = np.random.default_rng(seed)
    actions = build_action_table(cfg.num_bins, cfg.k_observed)
    action_index = {a: i for i, a in enumerate(actions)}
    env = make_env(cfg, source, seed=seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent = SchedulerAgent(cfg, len(actions), spec["recurrent"], spec["double"], device, rng)
    builder, shaper = StateBuilder(cfg), RewardShaper(cfg)
    explore = EpsilonGuidedExploration(cfg, action_index, rng)

    ckpt_path = os.path.join(out_dir, f"checkpoint_{variant}.pt")

    
    total_steps = 0
    start_ep = 1
    hist = {"base_reward": [], "shaped_reward": [], "pd": [], "pfa": [], "scan_eff": [], "epsilon": [], "loss": []}

    if resume and os.path.exists(ckpt_path):
        total_steps, prev_episodes, loaded_hist = agent.load_checkpoint(ckpt_path)
        hist = loaded_hist  
        start_ep = prev_episodes + 1
        print(f"--> Resuming training from Episode {start_ep} (Total steps: {total_steps})")

    t0 = time.time()
    end_ep = start_ep + num_episodes - 1

    for ep in range(start_ep, end_ep + 1):
        obs = env.reset()                      
        builder.reset()
        shaper.reset_episode()
        stats, losses = RunningStats(), []
        state = builder.state()
        
        for _ in range(cfg.episode_length):
            total_steps += 1
            eps = explore.epsilon(total_steps)
            if total_steps <= cfg.warmup_steps or rng.random() < eps:
                a = explore.explore(builder)
            else:
                a = agent.greedy_action(*state)
            
            bins = actions[a]
            obs, base_r, done, info, _, _, _ = env.step(bins)
            shaped, _ = shaper.shape(base_r, a, obs["intercepted_pdws"])
            builder.observe(bins, obs)
            next_state = builder.state()
            
            agent.buffer.push(state[0], state[1], a, shaped, next_state[0], next_state[1], 0.0)
            state = next_state
            stats.add(info, cfg.k_observed, base_r, shaped)
            
            if total_steps > cfg.warmup_steps and total_steps % cfg.learn_every == 0:
                loss = agent.learn()
                if loss is not None:
                    losses.append(loss)
            if done:
                break
                
        m = stats.summary()
        for key in ("pd", "pfa", "scan_eff"):
            hist[key].append(m[key])
        hist["base_reward"].append(stats.base_reward)
        hist["shaped_reward"].append(stats.shaped_reward)
        hist["epsilon"].append(explore.epsilon(total_steps))
        hist["loss"].append(float(np.mean(losses)) if losses else float("nan"))
        
        # Periodically save full checkpoint state
        if ep % print_every == 0 or ep == end_ep:
            agent.save_checkpoint(ckpt_path, total_steps, ep, hist)

        if verbose and (ep % print_every == 0 or ep == end_ep):
            w = slice(-print_every, None)
            print(f"    [{variant} | seed {seed}] ep {ep:4d} | reward {np.nanmean(hist['base_reward'][w]):8.1f} | "
                  f"Pd {np.nanmean(hist['pd'][w]):.3f} | eff {np.nanmean(hist['scan_eff'][w]):.3f} | "
                  f"eps {hist['epsilon'][-1]:.2f} | {time.time() - t0:5.0f}s")
                  
    agent.buffer = None
    return {"agent": agent, "history": hist, "seed": seed, "variant": variant}


def plot_training(hist: Dict, path: str) -> None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a, key, ttl in ((ax[0], "base_reward", "Training reward"), (ax[1], "pd", "Training Pd (with exploration)")):
        y = np.nan_to_num(np.array(hist[key], float))
        w = min(20, len(y))
        a.plot(np.arange(len(y)), y, alpha=0.3)
        a.plot(np.arange(w - 1, len(y)), np.convolve(y, np.ones(w) / w, mode="valid"))
        a.set(title=ttl, xlabel="episode")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"  saved plot: {path}")


def save_frozen(result: Dict, cfg: Config, out_dir: str, mode: str, subset: str, source_kind: str) -> str:
    agent = result["agent"]
    path = os.path.join(out_dir, "dqn_scheduler_frozen.pt")
    freeze_network(agent.q_network, cfg, path, agent.recurrent, agent.double, agent.num_actions,
                   extra={"variant": result["variant"], "seed": result["seed"], "data_mode": mode,
                          "train_subset": subset, "data_source": source_kind})
    return path


def main():
    ap = argparse.ArgumentParser(description="Train the DQN scheduler on streamed TSRD pulse trains")
    add_data_args(ap, default_subset="train", default_mode="stare")
    ap.add_argument("--variant", choices=list(VARIANTS), default="recurrent_ddqn")
    ap.add_argument("--episodes", type=int, default=100)   # episode count
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--bins", type=int, default=10)
    ap.add_argument("--k", type=int, default=3)
    ap.add_argument("--freq-range", type=float, nargs=2, default=[2.0, 6.0], metavar=("LO_GHZ", "HI_GHZ"))
    ap.add_argument("--frame-window-us", type=float, default=10.0)
    ap.add_argument("--episode-length", type=int, default=200)
    ap.add_argument("--snr-db", type=float, default=10.0, help="nominal training SNR")
    ap.add_argument("--use-ekf", action="store_true")
    ap.add_argument("--out-dir", default="train_out")
    ap.add_argument("--probe-only", action="store_true")
    ap.add_argument("--quick", action="store_true", help="tiny run to smoke-test the pipeline")
    ap.add_argument("--no-resume", action="store_true", help="Start fresh, ignoring any existing saved checkpoints")
    a, _ = ap.parse_known_args()

    cfg = Config(num_bins=a.bins, k_observed=a.k, freq_range_ghz=tuple(a.freq_range), snr_db=a.snr_db,
                 frame_window_us=a.frame_window_us, episode_length=a.episode_length, use_ekf=a.use_ekf)
    if a.quick:
        cfg.episode_length, cfg.warmup_steps, cfg.eps_decay_steps = 60, 200, 800
        a.episodes = 8
    os.makedirs(a.out_dir, exist_ok=True)
    source = source_from_args(a, a.mode, a.subset)
    try:
        probe_source(source, cfg)
        if a.probe_only:
            return
        
        res = train(cfg, source, a.variant, a.seed, a.episodes, 
                    out_dir=a.out_dir, resume=not a.no_resume)
        
        plot_training(res["history"], os.path.join(a.out_dir, "training_curves.png"))
        ckpt = save_frozen(res, cfg, a.out_dir, a.mode, a.subset, a.source)
        print(f"\nTrain stream: {source.stats()}\nNext:  python evaluate.py --checkpoint {ckpt} "
              f"--source {a.source} --subset validation")
    finally:
        source.close()


if __name__ == "__main__":
    main()